## Testlauf mit Jupyter Notebook

### Importe

In [1]:
# importe und allgemeiner DocString

"""evaluation.py -- Bewertung und Vergleich generierter Peptidsequenzen.

Architektur (nach Torres et al., Eval-Teil):

    SequenceSet   Datencontainer: Name + DataFrame (eine Zeile pro Sequenz)
    Scorer        berechnet Pro-Sequenz-Spalten (ProGen2, ESM-PPPL, APEX, Entropie)
    Evaluator     fuehrt mehrere Scorer aus, mit Cache auf Platte
    Filter        SequenceSet -> SequenceSet (z.B. MIC-Filter)
    Comparison    vergleicht mehrere SequenceSets, liefert Tabelle + Figure

Datenfluss:

    SequenceSet --Evaluator(Scorer...)--> SequenceSet (mit Score-Spalten)
                --Filter--> gefiltertes SequenceSet
                --Comparison({"train":..., "generiert":..., "gefiltert":...})--> ComparisonResult

Vertrag zwischen den Teilen (NICHT ohne Absprache aendern):
    * Pflichtspalte jedes SequenceSet: "sequence" (str, Grossbuchstaben, 20 Standard-AS)
    * Scorer haengen nur Spalten an, sie veraendern oder loeschen keine.
    * Spaltennamen der Scorer (siehe Scorer.columns):
        ppl_progen2   Perplexitaet unter ProGen2 (kleiner = natuerlicher)
        pppl_esm2     Pseudo-Perplexitaet unter ESM2
        mic_apex      vorhergesagte MIC (APEX)
        aa_entropy    Shannon-Entropie der AS-Verteilung der Sequenz (in bit)
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from pathlib import Path
from typing import Mapping, Sequence, ClassVar
from tqdm.auto import tqdm
import hashlib

#Essentielle 
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import math 
import matplotlib.pyplot as plt
import statistics as stats

#Transformer
from transformers import AutoModelForCausalLM
from tokenizers import Tokenizer     
import esm     

from aguirre.config import Config, CACHE_DIR
from aguirre.data import read_fasta
from aguirre.helpers import _shannons_H, Akkumulator
cfg = Config()

AMINO_ACIDS = cfg.AS_alphabet

#### Allgemeine DataClass (SequenceSet)
Speichert Sequence Sets mit Namen und DataFrame ein. 


In [2]:
# ---------------------------------------------------------------------------
# 1. Datencontainer
# ---------------------------------------------------------------------------

@dataclass
class SequenceSet:
    """Eine benannte Menge von Sequenzen plus ihre berechneten Eigenschaften.

    Attributes:
        name: Anzeigename, wird in Plots und Tabellen als Gruppenlabel genutzt
            (z.B. "train", "generiert", "gefiltert").
        data: DataFrame mit mindestens der Spalte "sequence". Jede weitere
            Spalte ist eine Pro-Sequenz-Eigenschaft (Scorer-Ausgabe).
    """
    # Verhält sich wie eine __init__
    name: str
    data: pd.DataFrame
    PROTECTED: ClassVar[tuple[str, ...]] = ("sequence", "length")
    
    def __post_init__(self) -> None: #Gut, wenn Sequence fehlt, daher "nach der __init__"
        """Prueft, dass die Spalte "sequence" existiert und nur gueltige AS enthaelt."""
        if "sequence" not in self.data.columns:
            raise ValueError(f"Spalte 'sequence' fehlt, vorhanden: {list(self.data.columns)}")
        seqs = self.data["sequence"]
        if seqs.isna().any() or (seqs == "").any():
            raise ValueError("Leere oder fehlende Sequenzen im Set.")
        invalid = set("".join(seqs)) - set(AMINO_ACIDS)
        if invalid:
            raise ValueError(f"Ungueltige Zeichen in Sequenzen: {sorted(invalid)}")
        self.data = self.data.assign(length=seqs.str.len().astype(int))
        

    @classmethod
    def from_fasta(cls, path: Path, name: str) -> "SequenceSet":
        ids, sequences = read_fasta(path)
        data = pd.DataFrame({"id": ids, "sequence": sequences})
        return cls(name=name, data=data)

    @classmethod
    def from_csv(cls, path: Path, name: str, sequence_col: str = "sequence") -> "SequenceSet":
        data = pd.read_csv(path)
        if sequence_col != "sequence":
            data = data.rename(columns={sequence_col: "sequence"})
        return cls(name=name, data=data)

    def with_columns(self, new_cols: pd.DataFrame) -> "SequenceSet":
        if len(new_cols) != len(self.data):
            raise ValueError("new_cols hat eine andere Zeilenzahl als das Set.")
        # gleiche Spaltennamen werden ueberschrieben (z.B. bei erneutem Evaluator-Lauf)

        verletzt = [c for c in new_cols.columns if c in self.PROTECTED]
        if verletzt:
            raise ValueError(
                f"Geschuetzte Spalten {verletzt} duerfen nicht ueberschrieben werden "
                f"(geschuetzt: {list(self.PROTECTED)})."
            )

        old = self.data.drop(columns=new_cols.columns.intersection(self.data.columns))
        data = pd.concat([old, new_cols.set_axis(old.index)], axis=1)
        return SequenceSet(name=self.name, data=data)
        

    def subset(self, mask: pd.Series, name: str | None = None) -> "SequenceSet":
        if mask.dtype != bool or len(mask) != len(self.data):
            raise ValueError("mask muss ein bool-Series in der Laenge des Sets sein.")
        data = self.data.loc[mask.to_numpy()].reset_index(drop=True)
        return SequenceSet(name=name or self.name, data=data)

### APEX Backends

In [3]:
class APEXBackend(ABC):
    """Die EINZIGE Stelle im Projekt, die APEX kennt.

    Trennt den Scorer-Vertrag (DataFrame mit festen Spalten) von der
    konkreten APEX-API. Sieht die echte API anders aus als erwartet,
    aenderst du nur eine Unterklasse hiervon.

    Attributes:
        strains: Namen der Pathogenstaemme, in der Reihenfolge der Spalten
            von predict(). Legt die Spaltennamen des Scorers fest.
    """

    strains: list[str]

    @abstractmethod
    def predict(self, sequences: Sequence[str]) -> np.ndarray:
        """Vorhergesagte MIC-Werte.

        Returns:
            Array (len(sequences), len(self.strains)) in µmol/L,
            Zeilenreihenfolge wie die Eingabe. Alle Werte > 0.
        """


class DummyAPEXBackend(APEXBackend):
    """Platzhalter ohne APEX-Installation -- deterministischer Unsinn.

    Zweck: Pipeline (Evaluator, MICFilter, Comparison) vollstaendig bauen
    und testen, bevor APEX laeuft. Werte sind reproduzierbar und liegen im
    richtigen Wertebereich, haben aber KEINE biologische Bedeutung.
    """

    strains = [f"strain_{i:02d}" for i in range(11)]

    def predict(self, sequences: Sequence[str]) -> np.ndarray:
        out = np.empty((len(sequences), len(self.strains)), dtype=float)
        for r, seq in enumerate(sequences):
            seed = int(hashlib.sha1(seq.encode()).hexdigest()[:8], 16)
            rng = np.random.default_rng(seed)
            # MIC ist log-verteilt -> im log2-Raum ziehen, dann zurueck.
            # loc=5 entspricht einem Median um 2**5 = 32 µmol/L.
            out[r] = 2.0 ** rng.normal(loc=5.0, scale=1.5, size=len(self.strains))
        return out


### Scrorer

In [ ]:
class Scorer(ABC):
    """Basisklasse: nimmt Sequenzen, gibt ein DataFrame mit Score-Spalten zurueck.

    Ein Scorer kennt weder Vergleiche noch Plots. Er muss nur wissen, wie man
    aus einer Sequenz eine oder mehrere Zahlen macht.

    Attributes:
        name: eindeutiger Name, dient als Cache-Schluessel (z.B. "progen2").
        columns: Namen der Spalten, die score() zurueckgibt.
    """

    name: str
    columns: list[str]

        

    @property
    def fingerprint(self) -> str:
        """Cache-Identitaet. ALLES, was das Ergebnis veraendert, gehoert hinein.

        Default ist der blosse Name -- nur korrekt fuer parameterfreie,
        deterministische Scorer. Jeder Scorer mit einem Modell dahinter MUSS
        das ueberschreiben, sonst liefert der Cache nach einem Modellwechsel
        stillschweigend alte Zahlen.
        """
        return self.name
            

    @abstractmethod
    def score(self, sequences: Sequence[str]) -> pd.DataFrame:
        """Berechnet die Scores.

        Args:
            sequences: Sequenzen in beliebiger Reihenfolge.

        Returns:
            DataFrame mit len(sequences) Zeilen, gleicher Reihenfolge wie die
            Eingabe und genau den Spalten aus self.columns.
        """


class AminoAcidEntropyScorer(Scorer):
    """Shannon-Entropie der Aminosaeureverteilung pro Sequenz ("Residuenvielfalt").

    H = -sum_a p_a * log2(p_a), p_a = Anteil der AS a in der Sequenz.
    Niedrig = wenige AS dominieren, hoch = gleichmaessig gemischt (max. log2(20)).
    Braucht kein Modell, taugt daher als erster Test der Pipeline.
    """

    name = "aa_entropy"
    columns = ["aa_entropy"]

    def score(self, sequences: Sequence[str]) -> pd.DataFrame:
        H = _shannons_H(sequences)
        return pd.DataFrame({self.columns[0]: H})
            

class ProGen2Scorer(Scorer):
    """Perplexitaet unter ProGen2 (autoregressiv, trainiert auf natuerlichen Proteinen).

    PPL = exp(-1/N * sum_i log p(x_i | x_<i)).  Kleiner = natuerlicher.
    Laeuft auf GPU -> in Batches arbeiten, Modell nur einmal laden.
    """

    name = "progen2"
    columns = ["ppl_progen2"]

    @property
    def fingerprint(self) -> str:
        return f"progen2__{self.cfg.progen_model_name.replace('/', '-')}"

    def __init__(self, config: Config): # kein nn.Module, da Kein Training, keine Register Buffer, keine Checkpoints
        """Laedt Modell und Tokenizer (siehe Cluster-Anleitung: Gewichte vorab laden)."""
        super().__init__()
        self.cfg = config
        model = AutoModelForCausalLM.from_pretrained(
        self.cfg.progen_model_name, trust_remote_code=True
        )
        model.eval()
        model.requires_grad_(False)
        tokenizer = Tokenizer.from_pretrained(
            self.cfg.progen_model_name
        )
        self.model = model
        self.tokenizer = tokenizer
        
        #Token-Ids als Attribute
        self.bos_id = self.tokenizer.token_to_id("1")
        self.eos_id = self.tokenizer.token_to_id("2")
        self.pad_id = self.tokenizer.token_to_id("<|pad|>")

        assert self.bos_id is not None, f"ID für BOS nicht gefunden"
        assert self.eos_id is not None, f"ID für EOS nicht gefunden"
        assert self.pad_id is not None, f"ID für Padding-Position nicht gefunden"
        for AS in self.cfg.AS_alphabet:
            if self.tokenizer.token_to_id(AS) is None:
                raise KeyError(
                    f"{AS} aus dem Config-Alphabet {self.cfg.AS_alphabet} hat keine ID im Tokenizer."
                )
            
        self.batch_size = self.cfg.progen_batch_size
        self.device = self.cfg.device
        self.model.to(self.device)
    
    @torch.no_grad()
    def score(self, sequences: Sequence[str]) -> pd.DataFrame:
        ppl_akk = []
        for seq in tqdm(sequences):
            seq_adapt = "1" + seq + "2" # dim L+2
            encoded_seq = self.tokenizer.encode(seq_adapt).ids # dim L+2
            ids = torch.tensor(encoded_seq, dtype= torch.long, device = self.device).unsqueeze(0) # dim 1, L+2
            logits = self.model(ids).logits # dim 1, L+2, 30
            pred = logits[0,:-2,:] # dim  L, 30
            target = ids[0,1:-1] # 
            CE =F.cross_entropy(pred, target)
            PPL = torch.exp(CE).item()
            ppl_akk.append(PPL)
        return pd.DataFrame({self.columns[0]: ppl_akk})


class ESMPseudoPerplexityScorer(Scorer):
    """Pseudo-Perplexitaet unter ESM2 (maskiertes LM, Salazar et al. 2020).

    Jede Position wird einmal maskiert -> N Forward-Passes pro Sequenz.
    Alle maskierten Varianten einer Sequenz zu einem Batch buendeln.
    Zweitmetrik, weil der Generator selbst ESM2-Embeddings nutzt.
    """

    name = "esm2_pppl"
    columns = ["pppl_esm2"]

    @property
    def fingerprint(self) -> str:
        return f"esm2_pppl__{self.cfg.esm_scorer_model_name}"

    def __init__(self, config: Config) -> None:
        super().__init__()
        self.cfg = config
        self.device = self.cfg.device

        mode = cfg.esm_scorer_model_name
        assert mode in ("esm2_t33_650M_UR50D", "esm2_t6_8M_UR50D"), f"Unbekannter Modus '{mode}', erlaubt sind 'esm2_t33_650M_UR50D' und 'esm2_t6_8M_UR50D'"
        #==Auswahl Moden==
        if mode == "esm2_t33_650M_UR50D":
            model,alphabet = esm.pretrained.esm2_t33_650M_UR50D() # Tupel ESM2-Model und Tokentabelle
        elif mode == "esm2_t6_8M_UR50D":
            model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
        assert model.embed_dim == self.cfg.esm_scorer_embed_dim, f"ESM liefert {model.embed_dim}, Config sagt {self.cfg.esm_scorer_embed_dim}"
        
        model.eval()
        model.requires_grad_(False)
        model.to(self.device)

        self.batch_converter = alphabet.get_batch_converter()
        self.model = model
        self.mask_idx = alphabet.mask_idx
        self.cls_idx = alphabet.cls_idx
        self.eos_idx = alphabet.eos_idx
        
#Notiz: Das comparison sollte ein csv=True/False einbauen + die benennung der Dateien nach 
# Datensatz_name, Score und Plotart.
    @torch.no_grad #HIER WEITER: Es fehlt das Verständnis + Workflow
    def score(self, sequences: Sequence[str]) -> pd.DataFrame:
        pppl_akk = []
        for seq in sequences:
            _,_,tokens_orig = self.batch_converter([("seq", seq)])
            tokens_orig = tokens_orig.to(self.device) #Muss zugewiesen werden, weil .to eines neues objekt generiert
            assert tokens_orig.shape == (1, len(seq) + 2), f"Falsche Shape {tokens_orig.shape}, muss {(1, len(seq) + 2)} sein"
            log_ps = []
            for i in range(len(seq)):
                tokens = tokens_orig.clone()
                echte_id = tokens[0,i + 1].item() # Vor dem Maskieren echte ID lesen
                tokens[0, i + 1] = self.mask_idx # Maskieren der aktuellen Position
                out = self.model(tokens)
                logits = out["logits"] # dim 1, N+2, 33, kein Softmax über die 20 kanonischen AS
                lp = torch.log_softmax(logits[0, i + 1], dim = -1)[echte_id].item() # softmax über die logits, dann die logits von der maskierten Posoition
                log_ps.append(lp)
            assert len(log_ps) == len(seq), f"Dimensionskonflikt mit Sequenzlänge und Länge aus forward-schleife"
            mean = stats.mean(log_ps)
            esm_score = math.exp(-mean)
            pppl_akk.append(esm_score)
        return pd.DataFrame({self.columns[0]: pppl_akk})


class APEXScorer(Scorer):
    def __init__(
        self,
        backend: APEXBackend,
        batch_size: int = 256,
        inaktive_schwelle: float = 64.0,
    ) -> None:
        """
        Args:
            mic_cap: Obergrenze in µmol/L. Hoehere Vorhersagen gelten als
                "inaktiv" und werden vor der Mittelung darauf geklemmt.
                Noetig, weil APEX bei 1e6 µM saettigt (ReLU am Ausgang) --
                ungeklemmt dominieren diese Werte jedes Log-Mittel.
                128 = eine Verduennungsstufe ueber Torres' Schwelle von 64.
        """
        self.backend = backend
        self.batch_size = batch_size
        self.inaktiv_schwelle = inaktiv_schwelle
        self.columns = (
            ["mic_apex", "apex_n_inaktiv"]
            + [f"mic_apex__{s}" for s in backend.strains]
        )    

    @property
    def fingerprint(self) -> str:
        return f"apex__{type(self.backend).__name__}__inakt{self.inaktiv_schwelle:g}"

    def score(self, sequences: Sequence[str]) -> pd.DataFrame:
        if len(sequences) == 0:
            return pd.DataFrame(columns=self.columns)

        bloecke = []
        for start in range(0, len(sequences), self.batch_size):
            batch = list(sequences[start : start + self.batch_size])
            mic = np.asarray(self.backend.predict(batch), dtype=float)
            erwartet = (len(batch), len(self.backend.strains))
            if mic.shape != erwartet:
                raise ValueError(f"Backend lieferte {mic.shape}, erwartet {erwartet}.")
            bloecke.append(mic)

        mic = np.vstack(bloecke)

        n_inaktiv = (mic > self.inaktiv_schwelle).sum(axis=1).astype(float)
        n_inaktiv[np.isnan(mic).any(axis=1)] = np.nan
        

        data = {
            "mic_apex": geometrisches_mittel(mic),
            "apex_n_inaktiv": n_inaktiv,
        }
        for j, stamm in enumerate(self.backend.strains):
            data[f"mic_apex__{stamm}"] = mic[:, j]      # je Stamm ungeklemmt
        return pd.DataFrame(data, columns=self.columns)

    

### Evaluator

In [5]:
class Evaluator:
    """Fuehrt Scorer ueber ein SequenceSet aus, mit Cache auf Platte.

    Cache-Schluessel ist die Sequenz selbst (kollisionsfrei, lesbar, debuggbar),
    eine Datei je Scorer-Fingerabdruck. NaN-Ergebnisse werden NICHT gecacht:
    sie koennen eine deterministische Ablehnung sein oder die Folge eines
    Absturzes -- und einen Absturz will man nicht einbetonieren.

    Nicht parallel ausfuehren: zwei Prozesse auf derselben Cache-Datei
    verlieren Schreibvorgaenge.
    """

    def __init__(
        self,
        scorers: Sequence[Scorer],
        cache_dir: Path | None = None,
        flush_every: int = 2000,
    ) -> None:
        """
        Args:
            scorers: anzuwendende Scorer.
            cache_dir: Ordner fuer den Cache; None = kein Caching.
            flush_every: nach so vielen neu berechneten Sequenzen wird der
                Cache geschrieben. Begrenzt den Verlust bei einem Abbruch.
        """
        self.scorers = list(scorers)
        self.cache_dir = Path(cache_dir) if cache_dir is not None else None
        self.flush_every = flush_every

        belegt: dict[str, str] = {}
        for sc in self.scorers:
            for spalte in sc.columns:
                if spalte in belegt:
                    raise ValueError(
                        f"Spalte '{spalte}' wird von '{belegt[spalte]}' und "
                        f"'{sc.fingerprint}' belegt."
                    )
                belegt[spalte] = sc.fingerprint

    def run(self, seqset: SequenceSet) -> SequenceSet:
        sequences = seqset.data["sequence"].tolist()
        bloecke = [self._run_scorer(sc, sequences) for sc in self.scorers]
        return seqset.with_columns(pd.concat(bloecke, axis=1))

    # -- pro Scorer --------------------------------------------------------

    def _run_scorer(self, scorer: Scorer, sequences: list[str]) -> pd.DataFrame:
        cache = self._load_cache(scorer)

        # dedupliziert und in stabiler Reihenfolge -- dieselbe Sequenz wird
        # einmal gerechnet, auch wenn sie mehrfach im Set steht.
        fehlend = [s for s in dict.fromkeys(sequences) if s not in cache.index]
        if fehlend:
            cache = self._score_missing(scorer, fehlend, cache)

        # reindex stellt die EINGABEreihenfolge her, dupliziert Mehrfach-
        # vorkommen und setzt NaN fuer alles, was nicht im Cache steht.
        werte = cache.reindex(sequences)[list(scorer.columns)]
        return werte.reset_index(drop=True)

    def _score_missing(
        self, scorer: Scorer, fehlend: list[str], cache: pd.DataFrame
    ) -> pd.DataFrame:
        for start in range(0, len(fehlend), self.flush_every):
            block = fehlend[start : start + self.flush_every]
            neu = scorer.score(block)

            if len(neu) != len(block):
                raise ValueError(
                    f"{scorer.fingerprint}.score() gab {len(neu)} Zeilen fuer "
                    f"{len(block)} Sequenzen zurueck."
                )
            if list(neu.columns) != list(scorer.columns):
                raise ValueError(
                    f"{scorer.fingerprint}.score() lieferte {list(neu.columns)}, "
                    f"erwartet {list(scorer.columns)}."
                )

            neu = neu.set_axis(pd.Index(block, name="sequence"))
            neu = neu[~neu.isna().any(axis=1)]          # NaN nicht cachen
            neu["computed_at"] = pd.Timestamp.now().isoformat(timespec="seconds")

            cache = neu if cache.empty else pd.concat([cache, neu])
            self._save_cache(scorer, cache)
        return cache

    # -- Cache -------------------------------------------------------------

    def _cache_path(self, scorer: Scorer) -> Path:
        return self.cache_dir / f"{scorer.fingerprint}.parquet"

    def _load_cache(self, scorer: Scorer) -> pd.DataFrame:
        leer = pd.DataFrame(
            columns=list(scorer.columns) + ["computed_at"],
            index=pd.Index([], name="sequence", dtype="object"),
        )
        if self.cache_dir is None:
            return leer
        pfad = self._cache_path(scorer)
        if not pfad.is_file():
            return leer

        df = pd.read_parquet(pfad)
        fehlt = [c for c in scorer.columns if c not in df.columns]
        if fehlt:
            raise ValueError(
                f"Cache {pfad} passt nicht zu '{scorer.fingerprint}': "
                f"Spalten {fehlt} fehlen. Datei loeschen oder Fingerabdruck pruefen."
            )
        return df

    def _save_cache(self, scorer: Scorer, cache: pd.DataFrame) -> None:
        if self.cache_dir is None:
            return
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        ziel = self._cache_path(scorer)
        # Erst daneben schreiben, dann atomar umbenennen: ein Abbruch mitten
        # im Schreiben hinterlaesst sonst eine kaputte Cache-Datei.
        tmp = ziel.with_suffix(".parquet.tmp")
        cache.to_parquet(tmp)
        tmp.replace(ziel)

### Filter

In [ ]:
class Filter(ABC):
    """Basisklasse fuer Filter. Ein Filter waehlt Sequenzen aus, er berechnet nichts."""

    @abstractmethod
    def apply(self, seqset: SequenceSet) -> SequenceSet:
        """Gibt das gefilterte SequenceSet zurueck (neuer Name, z.B. "<name>_gefiltert")."""

class LengthFilter(Filter):
    """Behaelt Sequenzen mit min_len <= length <= max_len (Spalte "length")."""

    def __init__(self, min_len: int, max_len: int) -> None:
        if min_len > max_len:
            raise ValueError(f"min_len ({min_len}) groesser als max_len ({max_len}).")
        self.min_len = min_len
        self.max_len = max_len

    @classmethod
    def from_config(cls, cfg: Config) -> "LengthFilter":
        return cls(min_len=cfg.pep_min_len, max_len=cfg.pep_max_len)

    def apply(self, seqset: SequenceSet) -> SequenceSet:
        mask = seqset.data["length"].between(self.min_len, self.max_len)
        return seqset.subset(mask, name=f"{seqset.name}_len{self.min_len}-{self.max_len}")


class MICFilter(Filter):
    """Behaelt Sequenzen mit vorhergesagter MIC <= max_mic (Spalte "mic_apex")."""

    def __init__(self, max_mic: float, column: str = "mic_apex") -> None:
        self.max_mic = max_mic
        self.column = column

    @classmethod
    def from_config(cls, cfg: Config) -> "MICFilter":
        return cls(max_mic=cfg.apex_mic_schwelle)

    def apply(self, seqset: SequenceSet) -> SequenceSet:
        if self.column not in seqset.data.columns:
            raise KeyError(
                f"Spalte '{self.column}' fehlt in '{seqset.name}' -- "
                "erst den APEXScorer laufen lassen."
            )
        # NaN bedeutet "nicht bewertbar". NaN <= max_mic ist False, solche
        # Sequenzen fliegen also raus. Das ist Absicht: ungeprueft heisst
        # nicht bestanden.
        mask = seqset.data[self.column] <= self.max_mic
        return seqset.subset(mask, name=f"{seqset.name}_mic{self.max_mic:g}")

### Statistikhelfer

In [7]:
def geometrisches_mittel(werte: np.ndarray, axis: int = 1) -> np.ndarray:
    """Mittel im Log-Raum. Fuer log-verteilte Groessen wie MIC.

    NaN wird durchgereicht: eine Sequenz, die der Backend nicht bewerten
    konnte, bekommt auch kein Aggregat.
    """
    if (werte <= 0).any():          # NaN <= 0 ist False, stoert also nicht
        raise ValueError("Nur positive Werte (Log-Mittel).")
    return np.exp(np.log(werte).mean(axis=axis))

### Test ProGen2 Scorer

In [ ]:
from tokenizers import Tokenizer
tok = Tokenizer.from_pretrained("hugohrban/progen2-small")

print("Vokabulargröße:", tok.get_vocab_size())
for i in range(tok.get_vocab_size()):
    print(i, repr(tok.id_to_token(i)))

print("IDs von '1' und '2':", tok.token_to_id("1"), tok.token_to_id("2"))
print(tok.encode("1MKTLLV2").ids)

In [ ]:
sc = ProGen2Scorer(cfg)
print(sc.bos_id, sc.eos_id, sc.pad_id)
print(next(sc.model.parameters()).device, sc.model.training)
print(sc.tokenizer.encode("1" + "MKTLLV" + "2").ids)

In [ ]:
## Test ProGen2scorer 2
from aguirre.config import TRACKED_DIR

PATH = TRACKED_DIR / "torres_train_sequences.csv"
#Zufallsequenz.
import random 
AS = cfg.AS_alphabet

def zufallssequenz(laenge, alphabet= AS):
    """Baut Sequenz aus länge und Alphabet aus"""
    return "".join(random.choices(AS,k= laenge))

seq_1 = zufallssequenz(laenge= 20)

seq_2 = 20*"A"


human_UBI_seq = "MTVRNIASICNMGTNASALEKDIGPEQFPINEHYFGLVNFGNTCYCNSVLQALYFCRPFRENVLAYKAQQKKKENLLTCLADLFHSIATQKKKVGVIPPKKFISRLRKENDLFDNYMQQDAHEFLNYLLNTIADILQEEKKQEKQNGKLKNGNMNEPAENNKPELTWVHEIFQGTLTNETRCLNCETVSSDEDFLDLSVDVEQNTSITHCLRDFSNTETLCSEQKYYCETCCSKQEAQKRMRVKKLPMILALHLKRFKYMEQLHRYTKLSYRVVFPLELRLFNTSSDAVNLDRMYDLVAVVVHCGSGPNRGHYITIVKSHGFWLLFDDDIVEKIDAQAIEEFYGLTSDISKNSESGYILFYQSRE"
human_Insulin_seq = "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"
torres30 = pd.read_csv(PATH)["sequence"].head(30)

test_seqs = [seq_1, seq_2, human_UBI_seq, human_Insulin_seq]
test_df = pd.DataFrame({"sequence": test_seqs})
test_df = pd.concat(objs=[test_df, torres30], ignore_index= True)
test_list = list(test_df["sequence"])


sc_2 = ProGen2Scorer(config= cfg)
test_scores = sc_2.score(sequences= test_list)

test_df["score"]= test_scores
test_df["laengen"] = test_df["sequence"].str.len()

display(test_df)

In [ ]:
x,y = test_df["score"].to_numpy(), test_df["laengen"].to_numpy()

fig, ax = plt.subplots(1, 1, figsize = (5,4))
ax.scatter(x,y)
ax.set_xlabel("Score")
ax.set_ylabel("Sequenzlängen")
ax.set_yscale("log")
fig.show()

### Test ESM2-scorer

In [ ]:
cfg_test = Config()
cfg_test.esm_scorer_embed_dim = 320 
cfg_test.esm_scorer_model_name = "esm2_t6_8M_UR50D"
sc3 = ESMPseudoPerplexityScorer(cfg_test)
display(sc3.score(test_list))

### Test APEX

In [ ]:
sc4 = APEXScorer(DummyAPEXBackend())
df = sc4.score(["MKTLLV", "AAAAAAAAAA"])
print(df[["mic_apex"]])
print(sc4.score(["MKTLLV"])["mic_apex"].iloc[0])   # muss identisch sein -> deterministisch

### Evaluator Test

#### Erster Smoketest

In [ ]:
ev = Evaluator(
    scorers=[AminoAcidEntropyScorer(), APEXScorer(DummyAPEXBackend())],
    cache_dir=CACHE_DIR / "scores",
)

ss = SequenceSet(name="test", data=pd.DataFrame(
    {"sequence": ["MKTLLV", "AAAAAAAAAA", "MKTLLV"]}   # Duplikat mit Absicht
))

import time
t = time.perf_counter(); ss1 = ev.run(ss); print("1. Lauf", time.perf_counter()-t)
t = time.perf_counter(); ss2 = ev.run(ss); print("2. Lauf", time.perf_counter()-t)

print(ss1.data)
assert ss1.data.equals(ss2.data)                      # Cache aendert nichts
assert ss1.data.loc[0, "mic_apex"] == ss1.data.loc[2, "mic_apex"]   # Duplikat gleich

#### Zweiter Smoketest:

In [16]:
import random

from aguirre.config import Config, CACHE_DIR, TRACKED_DIR
from aguirre.apex_backends import ApexNativeBackend

cfg = Config()

backend = ApexNativeBackend.from_config(cfg)     # laedt 8 Modelle, dauert ein paar Sekunden
print(len(backend.models), "Modelle,", len(backend.strains), "Staemme, device:", backend.device)

# 8 echte, experimentell validierte AMPs aus dem Torres-Trainingsset
echte = pd.read_csv(TRACKED_DIR / "torres_train_sequences.csv")["sequence"].head(60).tolist()

# Negativkontrollen
random.seed(cfg.seed)
kontrollen = [20 * "A", "".join(random.choices(cfg.AS_alphabet, k=20))]

ss = SequenceSet(
    name="apex_smoke",
    data=pd.DataFrame({"sequence": echte + kontrollen}),
)

ev = Evaluator(
    scorers=[APEXScorer(backend, mic_cap=cfg.apex_inaktiv_schwelle)],
    cache_dir=CACHE_DIR / "scores",
)
res = ev.run(ss)

spalten = ["sequence", "length", "mic_apex", "apex_n_inaktiv"]
print(res.data[spalten].to_string())

n_echt = len(echte)
print(f"\nEchte AMPs  -- Median mean-MIC: {res.data['mic_apex'][:n_echt].median():.1f} µM")
print(f"Kontrollen  -- mean-MIC:        "
      f"{res.data['mic_apex'][n_echt:].round(1).tolist()} µM")
print(f"Anteil echter AMPs mit MIC <= 64 µM: "
      f"{(res.data['mic_apex'][:n_echt] <= 64).mean():.0%}")





8 Modelle, 11 Staemme, device: cpu
                                             sequence  length    mic_apex  apex_n_inaktiv
0                                       FFKKWPWWPWRRK      13  121.471021             9.0
1                           RSVCRQIKICRRRGGCYYLCTNRPY      25  128.000000            11.0
2         PKTLRKFFARIRGGRAAVLNALGKEEQIGRASNSGRKCARKKK      43  120.127125             8.0
3                                   SNTALRRYNQWATGHFM      17  128.000000            11.0
4                                           RCICVLGVC       9  128.000000            11.0
5                        CEWYNISCQLGNKGQWCTLTKECQRSCK      28  128.000000            11.0
6                                        TRAAVVLIVIRR      12  128.000000            11.0
7                                           IRRWKRVWW       9  107.756821             7.0
8                   LEGEVNKIALLSTNKAVVSLSNGVSVLTSKVLD      33  485.865833            11.0
9                             FLPLVRVAAKLIPSVVCAISKRC      23  29

In [ ]:
import pandas as pd
from Bio import SeqIO
from aguirre.config import Config, PROJECT_ROOT
from aguirre.apex_backends import ApexNativeBackend

cfg = Config()
APEX_REPO = PROJECT_ROOT / "external" / "apex-pathogen"

backend = ApexNativeBackend.from_config(cfg)
seqs = [str(r.seq) for r in SeqIO.parse(APEX_REPO / "test_seqs.fasta", "fasta")]

mic = pd.DataFrame(backend.predict(seqs), index=seqs, columns=backend.strains)

pd.set_option("display.width", 200)
print(mic.iloc[:, :4].round(2))
print("\nZeilen-Mittel (µM):", mic.mean(axis=1).round(1).tolist())
print("kleinster Einzelwert:", round(float(mic.to_numpy().min()), 3))